In [1]:
import pandas as pd
import numpy as np

loreto = pd.read_csv(
    "../data/processed/dengue_loreto.csv",
    dtype={"ubigeo": str, "edad": str, "diresa": str}
)
filas_iniciales = len(loreto)
loreto.shape

(107074, 11)

In [ ]:
loreto.keys()

Index(['departamento', 'provincia', 'distrito', 'ubigeo', 'enfermedad', 'ano',
       'semana', 'diresa', 'edad', 'tipo_edad', 'sexo'],
      dtype='str')

In [5]:
loreto["edad_num"] = pd.to_numeric(loreto["edad"], errors="coerce")
print(loreto["edad_num"].isna().sum(), "edades que no se pudieron convertir")

0 edades que no se pudieron convertir


In [8]:
loreto["edad_num"]

0         58
1         44
2         21
3         74
4         42
          ..
107069    39
107070    18
107071    53
107072     6
107073    18
Name: edad_num, Length: 107074, dtype: int64

In [13]:
factores = {"A": 1, "M": 1/12, "D": 1/365}

loreto["edad_anios"] = loreto["edad_num"] * loreto["tipo_edad"].map(factores)

In [14]:
loreto["edad_anios"]

0         58.0
1         44.0
2         21.0
3         74.0
4         42.0
          ... 
107069    39.0
107070    18.0
107071    53.0
107072     6.0
107073    18.0
Name: edad_anios, Length: 107074, dtype: float64

In [15]:
loreto["edad_anios"].describe()

count    107074.000000
mean         25.062328
std          17.916183
min           0.000000
25%          11.000000
50%          21.000000
75%          36.000000
max         104.000000
Name: edad_anios, dtype: float64

In [20]:
# Edades imposibles o muy improbables
print("Mayores de 100 años:", (loreto["edad_anios"] > 100).sum())

# Unidades raras: 12 meses o más debería registrarse en años, y 365 días o más, también
print("Meses >= 12:", ((loreto["tipo_edad"] == "M") & (loreto["edad_num"] >= 12)).sum())
print("Días >= 365:", ((loreto["tipo_edad"] == "D") & (loreto["edad_num"] >= 365)).sum())

Mayores de 100 años: 2
Meses >= 12: 1
Días >= 365: 0


In [21]:
loreto["edad_anios"].max()

np.float64(104.0)

In [22]:
loreto["grupo_edad"] = pd.cut(
    loreto["edad_anios"],
    bins=[0, 12, 18, 30, 60, np.inf],
    labels=["Niño (0-11)", "Adolescente (12-17)", "Joven (18-29)", "Adulto (30-59)", "Adulto mayor (60+)"],
    right=False
)
loreto["grupo_edad"].value_counts().sort_index()

grupo_edad
Niño (0-11)            27026
Adolescente (12-17)    18689
Joven (18-29)          24942
Adulto (30-59)         30739
Adulto mayor (60+)      5678
Name: count, dtype: int64

In [23]:
gravedad = {
    "DENGUE SIN SIGNOS DE ALARMA": "Sin signos de alarma",
    "DENGUE CON SIGNOS DE ALARMA": "Con signos de alarma",
    "DENGUE GRAVE": "Grave"
}

loreto["gravedad"] = pd.Categorical(
    loreto["enfermedad"].map(gravedad),
    categories=["Sin signos de alarma", "Con signos de alarma", "Grave"],
    ordered=True
)
loreto["gravedad"].value_counts(sort=False)

gravedad
Sin signos de alarma    90452
Con signos de alarma    16068
Grave                     554
Name: count, dtype: int64

In [24]:
loreto["con_alarma_o_grave"] = loreto["gravedad"] != "Sin signos de alarma"